###Contextualização e Desafio

Este notebook implementa um pipeline de sanitização de dados para os datasets de produtos e pedidos da Olist, utilizando apenas bibliotecas nativas do Python (`csv`, `re`, `datetime`). O objetivo é tratar inconsistências como dados ausentes, padronizar strings e aplicar regras de negócio específicas para garantir a qualidade dos dados para relatórios e modelos de Machine Learning.

###IMPORTAÇÃO DE BIBLIOTECAS

In [ ]:
import csv
import re
#import unicodedata
from datetime import datetime

###ABRINDO DATASETS

In [ ]:
with open("/content/drive/MyDrive/olist_orders_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    pedidos = list(leitor)
    print(pedidos[0].keys())
    print(len(pedidos))

dict_keys(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'])
99441


In [ ]:
with open("/content/drive/MyDrive/olist_products_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    print(leitor.fieldnames)
    produtos = list(leitor)
    print(len(produtos))

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
32951


###VALIDAÇÃO E TRATAMENTOS DE DADOS AUSENTES

In [ ]:
def tratar_dados_ausentes(produtos):

    categorias_corrigidas = 0
    dimensoes_corrigidas = 0

    campos_dimensoes = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]

    # ==========================
    # 1. Calcular médias
    # ==========================
    medias = {}

    for campo in campos_dimensoes:
        soma = 0
        quantidade = 0

        for produto in produtos:
            valor = produto[campo].strip()

            if valor != "":
                soma += float(valor)
                quantidade += 1

        medias[campo] = soma / quantidade if quantidade > 0 else 0

    # ==========================
    # 2. Corrigir dados ausentes
    # ==========================
    for produto in produtos:

        # Corrigir categoria
        if produto['product_category_name'].strip() == "":
            produto['product_category_name'] = "Sem Categoria"
            categorias_corrigidas += 1

        # Corrigir dimensões
        for campo in campos_dimensoes:

            if produto[campo].strip() == "":
                produto[campo] = str(round(medias[campo], 2))
                dimensoes_corrigidas += 1

    return (
        produtos,
        categorias_corrigidas,
        dimensoes_corrigidas
    )

###PADRONIZAÇÃO DE STRINGS E REGEX

In [ ]:
def padronizar_categorias(produtos):

    for produto in produtos:

        categoria = produto['product_category_name']

        categoria = categoria.lower().strip()

        #categoria = unicodedata.normalize('NFKD', categoria)
        #categoria = categoria.encode('ASCII', 'ignore').decode('ASCII')

        categoria = re.sub(r'[^a-z0-9_ ]', '', categoria)

        produto['product_category_name'] = categoria

    return produtos

###LÓGICA DE REGRA DE NEGÓCIO (FILTROS E VALIDAÇÃO)

In [ ]:
def validar_entregas_nulas(pedidos):

    entrega_nula_cancelado = 0
    entrega_nula_nao_cancelado = 0

    for pedido in pedidos:

        entrega = pedido[
            'order_delivered_customer_date'
        ]

        status = pedido['order_status']

        if entrega.strip() == "":

            if status == "canceled":
                entrega_nula_cancelado += 1

            else:
                entrega_nula_nao_cancelado += 1

    return (
        entrega_nula_cancelado,
        entrega_nula_nao_cancelado
    )

CHAMANDO A FUNÇÃO (VERIFICANDO HIPÓTESE DE DE NEGÓCIO)

In [ ]:
# Chamada da função
entrega_cancelado, entrega_nao_cancelado = validar_entregas_nulas(pedidos)

# Verificação da hipótese
if entrega_nao_cancelado == 0:
    print("Hipótese confirmada.")
else:
    print("Hipótese rejeitada.")

Hipótese rejeitada.


###FORTAMAÇÃO TEMPORAL (DATETIME)

In [ ]:
from datetime import datetime

def converter_datas(pedidos):

    datas_convertidas = 0

    for pedido in pedidos:

        data = pedido['order_approved_at'].strip()

        if data:

            try:
                data_obj = datetime.strptime(
                    data,
                    "%Y-%m-%d %H:%M:%S"
                )

                pedido['order_approved_at'] = (
                    data_obj.strftime("%d/%m/%Y")
                )

                datas_convertidas += 1

            except ValueError:
                pass

    return pedidos, datas_convertidas

###RELATÓRIO DE STATUS MANUAL

In [ ]:
def gerar_relatorio(
        total_produtos,
        total_pedidos,
        categorias_corrigidas,
        dimensoes_corrigidas,
        cancelados,
        datas_convertidas):

    print("\n===== RELATÓRIO FINAL ====")

    print(
        f"Produtos processados: "
        f"{total_produtos}"
    )

    print(
        f"Pedidos processados: "
        f"{total_pedidos}"
    )

    print(
        f"Categorias corrigidas: "
        f"{categorias_corrigidas}"
    )

    print(
        f"Dimensões corrigidas pela média: "
        f"{dimensoes_corrigidas}"
    )

    print(
        f"Datas convertidas: "
        f"{datas_convertidas}"
    )

    print(
        f"Pedidos cancelados "
        f"(entrega nula): "
        f"{cancelados}"
    )

    print("\nBase sanitizada com sucesso.")

###MAIN

In [ ]:
# Normaliza 'pedidos' caso tenha ficado como tupla
if (
    isinstance(pedidos, tuple)
    and len(pedidos) == 2
    and isinstance(pedidos[0], list)
):
    pedidos = pedidos[0]

# FASE 1
produtos, categorias_corrigidas, dimensoes_corrigidas = (
    tratar_dados_ausentes(produtos)
)

# FASE 2
produtos = padronizar_categorias(produtos)

# FASE 3
entrega_cancelado, entrega_nao_cancelado = (
    validar_entregas_nulas(pedidos)
)

# Exibir resultado da hipótese
if entrega_nao_cancelado == 0:
    print("\nHipótese confirmada.")
else:
    print(
        f"\nHipótese rejeitada. "
        f"{entrega_nao_cancelado} pedidos "
        f"possuem entrega nula e não estão cancelados."
    )

# FASE 4
pedidos, datas_convertidas = (
    converter_datas(pedidos)
)

# Total de pedidos cancelados
total_cancelados = sum(
    1
    for pedido in pedidos
    if pedido['order_status'] == 'canceled'
)

# FASE 5
gerar_relatorio(
    len(produtos),
    len(pedidos),
    categorias_corrigidas,
    dimensoes_corrigidas,
    total_cancelados,
    datas_convertidas
)


Hipótese rejeitada. 2346 pedidos possuem entrega nula e não estão cancelados.

===== RELATÓRIO FINAL ====
Produtos processados: 32951
Pedidos processados: 99441
Categorias corrigidas: 610
Dimensões corrigidas pela média: 8
Datas convertidas: 99281
Pedidos cancelados (entrega nula): 625

Base sanitizada com sucesso.
